In [0]:
-- TODO: manual review required — choose whether this migrated BTEQ script runs as a Databricks notebook, stored procedure, or Workflows DAG

-- Teradata index statistics removed because Delta has no indexes and Photon maintains table statistics automatically.
-- Teradata index statistics removed because Delta has no indexes and Photon maintains table statistics automatically.
-- Teradata index statistics removed because Delta has no indexes and Photon maintains table statistics automatically.
-- Routine Teradata column statistics collection removed because Photon maintains table statistics automatically.


/**************************************************************************\
* ADD NEW ACCOUNTS TO THE ACCOUNT SNAPSHOT                                 *
* ------------------------------------------------------------------------ *
* Transactions may arrive for accounts which are not yet delivered by      *
* KYCSYS (same day account opening, migrated books). A placeholder entry   *
* is created so that the transaction can be loaded; it is replaced by the  *
* full record with the next KYCSYS load (kycref.55). The account is also   *
* reported in the DQ log (dailytxn.07).                                    *
\**************************************************************************/

-- Source database qualifiers removed because no target Unity Catalog mapping was provided; these objects resolve in the current catalog and schema.
INSERT INTO ACCT_SNAPSHOT
(
       ACCT_ID,
       PRIMARY_PARTY_ID,
       PRDT_CD,
       SEGMENT_CD,
       ACCT_STATUS_CD,
       OPEN_DT,
       LAST_CUST_ACTV_DT,
       PLACEHOLDER_IND
)
SELECT dtl.ACCT_ID,
       NULL,
       NULL,
       'UNK',
       'U',
       MIN(dtl.BOOKING_DT),
       MAX(dtl.BOOKING_DT),
       'Y'
FROM DLYTXN_LOAD AS dtl
LEFT OUTER JOIN ACCT_SNAPSHOT AS snp
  ON snp.ACCT_ID = dtl.ACCT_ID
WHERE snp.ACCT_ID IS NULL
  AND TRIM(dtl.ACCT_ID) <> ''
GROUP BY dtl.ACCT_ID;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.


/**************************************************************************\
* DETERMINE FALLBACK FX RATES                                              *
* ------------------------------------------------------------------------ *
* The snapshot holds the latest published rate per currency. When the      *
* booking date differs from the snapshot rate date (back-dated postings,   *
* weekend / bank holiday bookings), the latest spot rate published on or   *
* before the booking date is used instead - looking back 10 days at most.  *
\**************************************************************************/

-- Converted the Teradata volatile table to a Databricks session-scoped temporary table; PRIMARY INDEX and ON COMMIT PRESERVE ROWS have no target equivalent.
-- TODO: manual review required — verify fixed-width CHAR(3) trailing-space behavior for currency-code joins.
CREATE TEMP TABLE VT_FX_FALLBACK
(
       CCY_CD      CHAR(3)        NOT NULL,
       BOOKING_DT  DATE           NOT NULL,
       RATE_DT     DATE,
       GBP_RATE    DECIMAL(18,8)
);
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- Source database qualifiers removed because no target Unity Catalog mapping was provided; these objects resolve in the current catalog and schema.
INSERT INTO VT_FX_FALLBACK
SELECT ccy.TXN_CCY,
       ccy.BOOKING_DT,
       fxr.RATE_DT,
       fxr.GBP_RATE
FROM
       (SELECT DISTINCT
               TXN_CCY,
               BOOKING_DT
        FROM DLYTXN_LOAD
        WHERE TXN_CCY <> 'GBP'
       ) AS ccy
INNER JOIN FX_RATE AS fxr
  ON fxr.CCY_CD = ccy.TXN_CCY
 AND fxr.RATE_TYP_CD = 'SP'
 AND fxr.RATE_DT <= ccy.BOOKING_DT
 AND fxr.RATE_DT >= ccy.BOOKING_DT - 10
 AND fxr.GBP_RATE > 0
QUALIFY ROW_NUMBER() OVER (
           PARTITION BY ccy.TXN_CCY, ccy.BOOKING_DT
           ORDER BY fxr.RATE_DT DESC
       ) = 1;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- Converted the deliberate post-load Teradata statistics refresh to Databricks column statistics.
ANALYZE TABLE VT_FX_FALLBACK COMPUTE STATISTICS FOR COLUMNS CCY_CD, BOOKING_DT;


/**************************************************************************\
* CONVERT TRANSACTION AMOUNTS TO GBP                                       *
* ------------------------------------------------------------------------ *
* GBP_RATE is expressed as units of currency per 1 GBP, hence              *
* GBP_AMT = TXN_AMT / GBP_RATE.                                            *
* RATE_FALLBACK_IND : N = snapshot rate of booking date                    *
*                     Y = fallback rate from FX history                    *
*                     X = no rate found (reported in dailytxn.07)          *
\**************************************************************************/

-- Source database qualifier removed because no target Unity Catalog mapping was provided; this object resolves in the current catalog and schema.
DELETE FROM DLYTXN_GBP;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- Source database qualifiers removed because no target Unity Catalog mapping was provided; these objects resolve in the current catalog and schema.
-- Teradata NULLIFZERO calls converted to NULLIF while preserving the explicit decimal quotient scale.
INSERT INTO DLYTXN_GBP
(
       TXN_REF,
       SUPPLIER_CD,
       TXN_CCY,
       TXN_AMT,
       RATE_DT,
       GBP_RATE,
       GBP_AMT,
       RATE_FALLBACK_IND
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       dtl.TXN_CCY,
       dtl.TXN_AMT,
       CASE
          WHEN dtl.TXN_CCY = 'GBP'          THEN dtl.BOOKING_DT
          WHEN snp.RATE_DT = dtl.BOOKING_DT THEN snp.RATE_DT
          ELSE fbk.RATE_DT
       END,
       CASE
          WHEN dtl.TXN_CCY = 'GBP'          THEN CAST(1 AS DECIMAL(18,8))
          WHEN snp.RATE_DT = dtl.BOOKING_DT THEN snp.GBP_RATE
          ELSE fbk.GBP_RATE
       END,
       CASE
          WHEN dtl.TXN_CCY = 'GBP'
          THEN dtl.TXN_AMT
          WHEN snp.RATE_DT = dtl.BOOKING_DT
          THEN CAST(dtl.TXN_AMT / NULLIF(snp.GBP_RATE, 0) AS DECIMAL(18,2))
          ELSE CAST(dtl.TXN_AMT / NULLIF(fbk.GBP_RATE, 0) AS DECIMAL(18,2))
       END,
       CASE
          WHEN dtl.TXN_CCY = 'GBP'          THEN 'N'
          WHEN snp.RATE_DT = dtl.BOOKING_DT THEN 'N'
          WHEN fbk.GBP_RATE IS NOT NULL     THEN 'Y'
          ELSE 'X'
       END
FROM DLYTXN_LOAD AS dtl
LEFT OUTER JOIN FX_RATE_SNAPSHOT AS snp
  ON snp.CCY_CD = dtl.TXN_CCY
LEFT OUTER JOIN VT_FX_FALLBACK AS fbk
  ON fbk.CCY_CD = dtl.TXN_CCY
 AND fbk.BOOKING_DT = dtl.BOOKING_DT;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- Teradata index statistics removed because Delta has no indexes and Photon maintains table statistics automatically.
-- Converted the deliberate pre-report Teradata statistics refresh to Databricks column statistics.
ANALYZE TABLE DLYTXN_GBP COMPUTE STATISTICS FOR COLUMNS RATE_FALLBACK_IND;


/**************************************************************************\
* REPORT FALLBACK RATE USAGE                                               *
* ------------------------------------------------------------------------ *
* Informational output for the batch log only.                             *
\**************************************************************************/

-- Source database qualifier removed because no target Unity Catalog mapping was provided; this object resolves in the current catalog and schema.
SELECT TXN_CCY,
       RATE_FALLBACK_IND,
       COUNT(*)     AS TXN_CNT,
       SUM(TXN_AMT) AS TXN_AMT,
       SUM(GBP_AMT) AS GBP_AMT
FROM DLYTXN_GBP
WHERE RATE_FALLBACK_IND <> 'N'
GROUP BY 1, 2
ORDER BY 1, 2;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- Converted cleanup to explicitly drop the session-scoped temporary table.
DROP TEMPORARY TABLE VT_FX_FALLBACK;
-- TODO: manual review required — replace this BTEQ error check with procedure SIGNAL handling or a Workflows failure branch.

-- BTEQ success exit removed; the Databricks runner controls job completion.

